In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from src.features import build_training_data

# Build the leakage-free training table from full history.
df = build_training_data(pd.read_csv("data/results.csv"))

# Time-based split: train on everything before 2022, test on 2022 onward.
CUTOFF = "2022-01-01"
train = df[df["date"] < CUTOFF]
test = df[df["date"] >= CUTOFF]

# Features the model learns from, and the label it predicts.
FEATURES = ["elo_diff", "neutral"]
X_train, y_train = train[FEATURES], train["outcome"]
X_test, y_test = test[FEATURES], test["outcome"]

# Multinomial logistic regression: three outcomes at once.
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("outcome classes:", list(model.classes_))
print("train rows:", len(train), "| test rows:", len(test))
print("test accuracy:", round(model.score(X_test, y_test), 4))


outcome classes: ['draw', 'home_loss', 'home_win']
train rows: 44837 | test rows: 4588
test accuracy: 0.5985


In [2]:
import numpy as np

# Three imaginary neutral-ground matches: strong favourite, even, big underdog.
sample = pd.DataFrame({"elo_diff": [300, 0, -300], "neutral": [1, 1, 1]})
probs = model.predict_proba(sample)

for diff, p in zip(sample["elo_diff"], probs):
    row = dict(zip(model.classes_, np.round(p, 3)))
    print(f"elo_diff={diff:>4}  ->  {row}")

elo_diff= 300  ->  {'draw': np.float64(0.135), 'home_loss': np.float64(0.054), 'home_win': np.float64(0.811)}
elo_diff=   0  ->  {'draw': np.float64(0.25), 'home_loss': np.float64(0.338), 'home_win': np.float64(0.411)}
elo_diff=-300  ->  {'draw': np.float64(0.168), 'home_loss': np.float64(0.757), 'home_win': np.float64(0.075)}


In [3]:
import joblib

# Bundle the model with its feature list and cutoff, so whatever loads it
# knows exactly how it was trained.
artifact = {
    "model": model,
    "features": FEATURES,
    "cutoff": CUTOFF,
    "classes": list(model.classes_),
}
joblib.dump(artifact, "src/model.joblib")
print("Saved src/model.joblib")

Saved src/model.joblib
